In [1]:
import config as config
import os
import pandas as pd
import ast

import misc
from models.logistic_regression import run_logistic_regression_models
from data_preproc import data_preproc_config

from data_preproc.data_preproc_functions import  Logger

logger = Logger(output_filename=None, suppress_CLI_prints=False)

c:\Users\d.c.macrae\.conda\envs\HNC_310\lib\site-packages\torch\cuda\__init__.py:118: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at ..\c10\cuda\CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
export_MDACC_CITOR_dir = "experiments/MDACC_CITOR"
os.makedirs(export_MDACC_CITOR_dir, exist_ok=True)

In [3]:
config.exp_root_dir = "c:\\Users\\d.c.macrae\\Documents\\DL_NTCP_Multitox\\experiments\\TransRPDieT_DenseNet121_heads_m2"

In [4]:
import os

fold_folders = [folder for folder in os.listdir(config.exp_root_dir) if os.path.isdir(os.path.join(config.exp_root_dir, folder))]
fold_folders

['20240709_161259_1_101_47_TransRP_DenseNet121_heads_m2_params_40798213_auc_tr_0.757_val_0.719_test_0.723',
 '20240709_161259_2_101_33_TransRP_DenseNet121_heads_m2_params_40798213_auc_tr_0.741_val_0.73_test_0.74',
 '20240709_161259_3_101_38_TransRP_DenseNet121_heads_m2_params_40798213_auc_tr_0.756_val_0.717_test_0.733',
 '20240709_161259_4_101_51_TransRP_DenseNet121_heads_m2_params_40798213_auc_tr_0.763_val_0.723_test_0.743',
 '20240709_161259_5_101_47_TransRP_DenseNet121_heads_m2_params_40798213_auc_tr_0.754_val_0.751_test_0.734_avg_tr_0.754_val_0.728_test_0.735_ens_0.74']

In [5]:
import torch
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
import json


from sklearn.utils._testing import ignore_warnings
from sklearn.exceptions import ConvergenceWarning
@ignore_warnings(category=ConvergenceWarning)
def run_logistic_regression_models_2(config, patient_id_col, logger):
    """
    Run logistic regression model on training (training + internal_validation) and test data.

    Note: if submodels_features and lr_coefficients are both None, then the logistic regression model will be
        directly fitted to features.

    Assumption: there are no overlapping of patient_ids between train, internal_validation and test set
        (see load_data.py).

    Args:
        config (Config): configuration object
        patient_id_col (str): name of the patient_id column
        submodels_features_list (list): list of lists of features for submodels
        features_list (list): list of features
        lr_coefficients_list (list): list of coefficients for submodels
        logger (Logger): logger object

    Returns:

    """   
    # submodels_features_list = config.submodels_features_list
    # features_lr_list = config.features_lr_list
    # lr_coefficients_list = config.lr_coefficients_list

    features_csv = os.path.join(config.data_dir, config.filename_stratified_sampling_test_csv)
    endpoint_list = config.endpoint_list
    patient_id_length = config.patient_id_length

    valid_endpoint_values = config.valid_endpoint_values
    seed = config.seed
    nr_of_decimals = config.nr_of_decimals

    patient_ids_json = os.path.join(config.exp_dir, config.filename_train_val_test_patient_ids_json)


    # Initialize variables
    test_y_pred_dict, test_y_dict = dict(), dict()

    # Load features + labels
    df = pd.read_csv(features_csv, sep=';', decimal='.')
    df[patient_id_col] = ['%0.{}d'.format(patient_id_length) % int(x) for x in df[patient_id_col]]

    learned_lr_coefficients = {}


    #df.loc[:, patient_id_col] = df.loc[:, patient_id_col].apply(lambda x: '%0.{}d'.format(patient_id_length) % int(x))

    for i, endpoint in enumerate(endpoint_list):
        logger.my_print('Fitting LR models for: {}.'.format(endpoint))
        
        endpoint_tox = endpoint.split("_")[0] # get only the name of the toxicty, not the endpoint time (e.g. only "Dysphagia", no "M06")
        # use that key to access the features lists. They are the same for all 'late' (post-treatment) CITOR models
        submodels_features = config.submodels_features_list_dict[endpoint_tox]
        features = config.features_lr_list_dict[endpoint_tox]
        if not config.use_umcg:
            features = [x.replace("W01", "BSL") for x in features]
        lr_coefficients = config.lr_coefficients_list_dict[endpoint_tox]

        # Initialize variables
        model = LogisticRegression(random_state=seed, C=1e10)
        test_patient_ids, test_y_pred, test_y = [None] * 3

        df_test = df

        # Test set
        test_patient_ids = df_test.loc[:, patient_id_col].tolist()
        test_X = df_test.loc[:, features]
        #print(test_X)
        test_y = df_test.loc[:, endpoint]
        # n_test = len(test_y)
        # n_test_0 = sum(test_y == 0)
        # n_test_1 = sum(test_y == 1)
        # assert n_test == n_test_0 + n_test_1

        #print(lr_coefficients)
        model.fit(test_X, test_y)
        if lr_coefficients is not None:
            model.intercept_ = np.array([lr_coefficients[0]])
            model.coef_ = np.array([lr_coefficients[1:]])
            model.classes_ = np.array([0, 1])

        #print(model.intercept_, model.coef_)
        #print(model)

        #test_y_pred = model.predict_proba(test_X)[:,1]
        test_y_pred = model.predict_proba(test_X)[:,1]
        test_y_pred = torch.tensor(test_y_pred, device=config.device)
        test_y = torch.tensor(test_y.tolist(), dtype=torch.int, device=config.device)
        # logger.my_print('Test size (label=0): {}/{} ({}).'.format(n_test_0, n_test,
        #                                                           round(n_test_0 / n_test, nr_of_decimals)))
        # logger.my_print('Test size (label=1): {}/{} ({}).'.format(n_test_1, n_test,
        #                                                           round(n_test_1 / n_test, nr_of_decimals)))

        test_y_pred_dict[endpoint] = test_y_pred
        test_y_dict[endpoint] = test_y
        learned_lr_coefficients[endpoint] = lr_coefficients

    return (test_patient_ids, test_y_pred_dict, test_y_dict,
            learned_lr_coefficients)


In [6]:
ensemble_DL_predictions = {endpoint: None for endpoint in config.endpoint_list}


for idx, folder in enumerate(fold_folders):
    test_y_pred_list_dict = dict()

    exp_dir = os.path.join(config.exp_root_dir, folder)
    results_dir = os.path.join(exp_dir, "results.csv")

    config.exp_dir = exp_dir
    
    df = pd.read_csv(results_dir, sep=";", index_col=0)

    lr_coeff_dict = ast.literal_eval(df.loc["lr_coefficients"].item())
    #
    #print("Done with folder: ", folder)
    

    for endpt in config.endpoint_list:
        new_key = endpt.split("_")[0]

        lr_coeff_dict[new_key] = lr_coeff_dict.pop(endpt)


    config.lr_coefficients_list_dict = lr_coeff_dict

    (test_patient_ids, test_y_pred_dict, test_y_dict, \
            learned_lr_coefficients) = run_logistic_regression_models_2(config, patient_id_col = data_preproc_config.patient_id_col, logger=logger)
    
    test_y_pred_list_dict = test_y_pred_dict
    

    for endpoint in config.endpoint_list:
        if ensemble_DL_predictions[endpoint] is None:
            ensemble_DL_predictions[endpoint] = [x for x in test_y_pred_list_dict[endpoint]]
        else:
            ensemble_DL_predictions[endpoint] = [x+y for x,y in zip(ensemble_DL_predictions[endpoint], test_y_pred_list_dict[endpoint])]

    test_y_true_list_dict = test_y_dict


    # save the predictions of this fol
    mode_list = ['test' for _ in range(len(test_patient_ids))]
    misc.save_predictions(config=config, patient_ids=test_patient_ids, endpoint_list=config.endpoint_list,
                                    y_pred_list_dict=test_y_pred_list_dict, y_true_list_dict=test_y_true_list_dict,  # test_y_true_list_dict should be the same for all folds!
                                    mode_list=mode_list, model_name=f'CITOR_external_fold_{idx}',
                                    exp_dir=export_MDACC_CITOR_dir, logger=logger) 
    
misc.save_predictions(config=config, patient_ids=test_patient_ids, endpoint_list=config.endpoint_list,
                                    y_pred_list_dict=ensemble_DL_predictions, y_true_list_dict=test_y_true_list_dict,  # test_y_true_list_dict should be the same for all folds!
                                    mode_list=mode_list, model_name='CITOR_external_ens',
                                    exp_dir=export_MDACC_CITOR_dir, logger=logger) 


2024-07-24 07:59:40 - INFO: Fitting LR models for: Aspiration_M06.


2024-07-24 07:59:40 - INFO: Fitting LR models for: Dysphagia_M06.
2024-07-24 07:59:40 - INFO: Fitting LR models for: Sticky_M06.
2024-07-24 07:59:40 - INFO: Fitting LR models for: Taste_M06.


[-3.495352753083858, 0.0027778233335600568, 1.9847109493331736, 2.5709446063534354]
[-3.49535275] [[0.00277782 1.98471095 2.57094461]]
[-2.700817102031551, 0.02060473230541541, 0.014729561470663712, 0.0069413407245966004, 0.01349227919433637, 1.034543323764006, 1.1710808384001612, -1.015165564502388, -1.5194958394835005]
[-2.7008171] [[ 0.02060473  0.01472956  0.00694134  0.01349228  1.03454332  1.17108084
  -1.01516556 -1.51949584]]
[-2.2719982100372227, 0.029603654944416326, -0.010975483694151028, 0.8125639405366417, 1.6431656851102379]
[-2.27199821] [[ 0.02960365 -0.01097548  0.81256394  1.64316569]]
[-4.894876749046384, -0.0005071242949461228, 0.04053195647132415, 3.8596782859780383]


2024-07-24 07:59:40 - INFO: Fitting LR models for: Xerostomia_M06.
2024-07-24 07:59:40 - INFO: Fitting LR models for: Aspiration_M06.
2024-07-24 07:59:40 - INFO: Fitting LR models for: Dysphagia_M06.
2024-07-24 07:59:40 - INFO: Fitting LR models for: Sticky_M06.


[-4.89487675] [[-5.07124295e-04  4.05319565e-02  3.85967829e+00]]
[-2.2033748543421146, 0.020358528942168497, 0.023013455295566646, 0.858176292715495, 1.67330296035845]
[-2.20337485] [[0.02035853 0.02301346 0.85817629 1.67330296]]
[-4.082277799277964, 0.022282980470760768, 1.1535285430238509, 1.859886401139742]
[-4.0822778] [[0.02228298 1.15352854 1.8598864 ]]
[-3.3571800736114144, 0.02570056154522635, 0.02370568999276848, 0.00822647980672375, 0.007262063721840691, 0.8275995448028062, 1.0409462164169485, -0.8296483244011927, -0.9453244138053722]
[-3.35718007] [[ 0.02570056  0.02370569  0.00822648  0.00726206  0.82759954  1.04094622
  -0.82964832 -0.94532441]]
[-2.4440344809649113, 0.03191282040407314, -0.009500920224657048, 0.7685509731248996, 1.647409449149751]


...

[-3.835592368165565, 0.01874678655628549, 1.3668347911582892, 2.0121366057479895]
[-3.83559237] [[0.01874679 1.36683479 2.01213661]]
[-3.0143216362042455, 0.019481087757335647, 0.016985437946656353, 0.006029481060885904, 0.012228720417168366, 0.9698830739144758, 1.1191374280782465, -0.6129458452230342, -1.2666100589869276]
[-3.01432164] [[ 0.01948109  0.01698544  0.00602948  0.01222872  0.96988307  1.11913743
  -0.61294585 -1.26661006]]
[-2.252302116468206, 0.027830633873483966, -0.0026126149347623865, 0.7272638048027102, 1.3335591224500376]


2024-07-24 07:59:41 - INFO: Fitting LR models for: Taste_M06.
2024-07-24 07:59:41 - INFO: Fitting LR models for: Xerostomia_M06.
2024-07-24 07:59:41 - INFO: Fitting LR models for: Aspiration_M06.
2024-07-24 07:59:41 - INFO: Fitting LR models for: Dysphagia_M06.


[-2.25230212] [[ 0.02783063 -0.00261261  0.7272638   1.33355912]]
[-4.5684410622754354, 0.0003143260227690542, 0.03840161895673886, 3.267650531832564]
[-4.56844106] [[3.14326023e-04 3.84016190e-02 3.26765053e+00]]
[-2.425971660503112, 0.021920069833125902, 0.025103801341755338, 0.876759947600878, 1.6051552263451603]
[-2.42597166] [[0.02192007 0.0251038  0.87675995 1.60515523]]
[-3.7962093810943567, 0.017385436482469787, 1.4095567379923926, 2.0163277747128903]
[-3.79620938] [[0.01738544 1.40955674 2.01632777]]


2024-07-24 07:59:41 - INFO: Fitting LR models for: Sticky_M06.
2024-07-24 07:59:41 - INFO: Fitting LR models for: Taste_M06.
2024-07-24 07:59:41 - INFO: Fitting LR models for: Xerostomia_M06.


[-3.1006146165814386, 0.019610542252118793, 0.015901246819326746, 0.011054177712368895, 0.00988722609726152, 1.0752845642922813, 1.2143746053178015, -0.6671346550153028, -1.2506228897040141]
[-3.10061462] [[ 0.01961054  0.01590125  0.01105418  0.00988723  1.07528456  1.21437461
  -0.66713466 -1.25062289]]
[-2.1732214515234642, 0.031194325577168846, -0.01750544337852245, 0.7584199137247684, 1.5820811985327046]
[-2.17322145] [[ 0.03119433 -0.01750544  0.75841991  1.5820812 ]]
[-4.29975749957708, -0.006705101292097611, 0.04237353562294522, 3.008236915849605]
[-4.2997575] [[-0.0067051   0.04237354  3.00823692]]
[-2.3299112479148727, 0.020556865285757713, 0.02402974596044958, 1.0077979243248982, 1.710214807132361]
[-2.32991125] [[0.02055687 0.02402975 1.00779792 1.71021481]]


In [13]:
from utils.metrics import compute_AUCs_for_multiple_endpoints

LR_ens_avg_auc_value, LR_ens_auc_value_dict = compute_AUCs_for_multiple_endpoints(config, ensemble_DL_predictions, test_y_true_list_dict)

In [14]:
from utils.calibration_plots import adaptive_make_calibration_plots
ensemble_calibration_plot_dict_list = [
    {
        "name" : "DL Model",
        "preds" : test_y_pred_list_dict,
        "labels" : test_y_true_list_dict,
    } #,{                                                                   # TODO: external validation CITOR
    #     "name" : "CITOR",
    #     "preds" : ensemble_LR_predictions,
    #     "labels" : test_y_lr_list_dict,
    # }
]
adaptive_make_calibration_plots(config, ensemble_calibration_plot_dict_list, column_names = config.endpoint_list, mode='calibration', 
                                title="Calibration Plot: Ensemble (Test Set)", filedir = os.path.join(export_MDACC_CITOR_dir, 'calibration_plot_external_validation_ens.png'))
[LR_test_ECE], [LR_test_MCE] = adaptive_make_calibration_plots(config, ensemble_calibration_plot_dict_list, column_names = config.endpoint_list, mode='reliability',
                                title="Reliability Plot: Ensemble (Test Set)", filedir = os.path.join(export_MDACC_CITOR_dir, 'reliability_plot_external_validation_ens.png'))

In [15]:
df.loc["lr_coefficients"].item()

"{'Aspiration_M06': [-3.7962093810943567, 0.017385436482469787, 1.4095567379923926, 2.0163277747128903], 'Dysphagia_M06': [-3.1006146165814386, 0.019610542252118793, 0.015901246819326746, 0.011054177712368895, 0.00988722609726152, 1.0752845642922813, 1.2143746053178015, -0.6671346550153028, -1.2506228897040141], 'Sticky_M06': [-2.1732214515234642, 0.031194325577168846, -0.01750544337852245, 0.7584199137247684, 1.5820811985327046], 'Taste_M06': [-4.29975749957708, -0.006705101292097611, 0.04237353562294522, 3.008236915849605], 'Xerostomia_M06': [-2.3299112479148727, 0.020556865285757713, 0.02402974596044958, 1.0077979243248982, 1.710214807132361]}"

In [19]:
results_dict = {
    "avg_ens_AUC": LR_ens_avg_auc_value,
    "ens_AUC": LR_ens_auc_value_dict,
    "ens_ECE": LR_test_ECE,
    "ens_MCE": LR_test_MCE,
    #"lr_coefficients": df.loc["lr_coefficients"].item()
}

config.exp_dir = export_MDACC_CITOR_dir
misc.create_ensemble_results(config, **results_dict)